# Operations: submit now, await later

Every call that puts Looper's GPUs to work returns twice: once when the service has accepted it, and once when it has run. In this tutorial you step a run one await at a time, then step it again with every call submitted ahead, and compare how long each takes. You also see where an error surfaces.

It uses the SDK alone, and the batch of [Your first fine-tune](102_first_sft.ipynb). Put your API key in the settings cell's `API_KEY`.

## Install the SDK and tokenizer

In [1]:
%pip install -q "goaly-looper==0.1.0b4" "transformers==5.19.0"

## Settings

The model, its tokenizer, where Looper is, your project and your API key, then how many steps each way of stepping takes.

In [2]:
MODEL = "Qwen/Qwen3.5-9B"
TOKENIZER = "Qwen/Qwen3.5-9B"
BASE_URL = "https://api.looper.goaly.ai"
PROJECT = "example"
API_KEY = "YOUR_API_KEY"
STEPS = 8
MAX_SEQ_LEN = 512

## A batch to train on

Eight question-and-answer pairs as datums: weight 0 on each question, 1 on its answer and the end token.

In [3]:
import time

from transformers import AutoTokenizer

from looper_sdk import CrossEntropy, Datum, EmptyStep, Lora, ServiceClient, TrainingConfig

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER)

PAIRS = [
    ("What is the capital of France?", "Paris."),
    ("What is two plus three?", "Five."),
    ("What color is a clear daytime sky?", "Blue."),
    ("How many legs does a spider have?", "Eight."),
    ("What is the opposite of hot?", "Cold."),
    ("Which planet do we live on?", "Earth."),
    ("What do bees make?", "Honey."),
    ("How many days are in a week?", "Seven."),
]

data = []
for question, answer in PAIRS:
    prompt = tokenizer.encode(f"Question: {question}\nAnswer:", add_special_tokens=False)
    reply = tokenizer.encode(f" {answer}", add_special_tokens=False) + [tokenizer.eos_token_id]
    data.append(Datum(tokens=(*prompt, *reply), weights=(0,) * len(prompt) + (1,) * len(reply)))

config = ServiceClient.Config(
    base_url=BASE_URL, project=PROJECT, api_key=API_KEY, display_name="operations"
)
training = TrainingConfig(
    model=MODEL,
    kind=Lora(rank=32),
    tokens_per_step=max(1024, MAX_SEQ_LEN, sum(len(datum.tokens) for datum in data)),
    max_seq_len=MAX_SEQ_LEN,
)
print(len(data), "datums")

8 datums


## Two awaits

- `await training_client.forward_backward(...)` returns an `Operation` once the service has stored the call and given it its place in the run's order, its `seq`. That takes one round trip.
- `await operation.result()` returns what the call computed, once it has run.

This first way of stepping awaits every result before it submits the next call. While your program waits for a result and sends the next call, the GPUs have nothing to do.

In [4]:
async def step_by_step(training_client, steps):
    results = []
    for _ in range(steps):
        backward = await training_client.forward_backward(data, CrossEntropy())
        await backward.result(timeout_s=300)
        update = await training_client.optim_step()
        results.append(await update.result(timeout_s=300))
    return results

## Submitting ahead

The service runs a run's operations in the order it accepted them. So nothing has to be awaited before the next call is submitted: an `optim_step` applies the `forward_backward` calls accepted before it, whenever they run.

This second way submits every call first and awaits the results afterwards. The run always has its next operation waiting.

In [5]:
async def submitted_ahead(training_client, steps):
    updates = []
    for _ in range(steps):
        await training_client.forward_backward(data, CrossEntropy())
        updates.append(await training_client.optim_step())
    return [await update.result(timeout_s=300) for update in updates]

## An error surfaces at its result

The service accepts an operation before it runs, so an operation that cannot succeed fails later, when `result()` is awaited. An `optim_step` with no gradients accumulated is the simplest case: it is accepted like any other and fails with `EmptyStep`.

A failed operation does not fail the ones accepted after it. This is why submitting ahead needs a limit in a real loop: when a step fails, the steps already submitted still run.

In [6]:
async def an_empty_step(training_client):
    update = await training_client.optim_step()
    print(f"accepted: {update.kind}, seq {update.seq}")
    try:
        await update.result(timeout_s=300)
    except EmptyStep as error:
        print("EmptyStep:", error.message)
    print("state:", (await update.state()).value)

## Run all three

One session and one run serve the whole comparison. A run's first step is slower than the rest, so one step is taken before the clock starts.

In [7]:
async with await config.build() as service:
    print("starting the run...")
    training_client = await service.create_training_client(training, timeout_s=20 * 60)
    await step_by_step(training_client, 1)

    started = time.perf_counter()
    await step_by_step(training_client, STEPS)
    print(f"step by step:    {STEPS} steps in {time.perf_counter() - started:.1f} s")

    started = time.perf_counter()
    results = await submitted_ahead(training_client, STEPS)
    print(f"submitted ahead: {STEPS} steps in {time.perf_counter() - started:.1f} s")

    await an_empty_step(training_client)

    # The run goes on after the failed step.
    await training_client.forward_backward(data, CrossEntropy())
    last = await (await training_client.optim_step()).result(timeout_s=300)
    print("the run is at step", last.step)

starting the run...
step by step:    8 steps in 8.1 s
submitted ahead: 8 steps in 6.7 s
accepted: optim_step, seq 35
EmptyStep: Nothing of nonzero weight was accumulated for this step: FAILED_PRECONDITION: EMPTY_STEP: no gradient accumulated since the last step
state: failed
the run is at step 18


## What the steps reported

The steps ran in the order they were accepted, so the results of the steps submitted ahead are in step order. Each `StepResult` holds the step's number, its loss sum and target count, its learning rate and its gradient norm.

In [8]:
for result in results:
    loss = result.loss_sum / result.weighted_targets
    print(f"step {result.step:2d}: loss {loss:.4f}, grad norm {result.grad_norm:.3f}")

step 10: loss 0.0023, grad norm 0.091
step 11: loss 0.0013, grad norm 0.052
step 12: loss 0.0008, grad norm 0.032
step 13: loss 0.0005, grad norm 0.034
step 14: loss 0.0003, grad norm 0.015
step 15: loss 0.0001, grad norm 0.006
step 16: loss 0.0001, grad norm 0.003
step 17: loss 0.0000, grad norm 0.002


## Next steps

Submitting ahead is how a training loop keeps the GPUs busy while your program prepares the next batch. A real loop submits a few steps ahead, not all of them, so a failure leaves only those few accepted.

- [The SFT loop](301_sft_loop.ipynb) uses the cookbook's loop, which does exactly that.
- [Chat data](201_chat_data.ipynb) turns conversations into datums.